# Notebook 02 — Data cleaning 

1. Objectives

This notebook applies the cleaning decisions established during profiling (Notebook 01) to the four ENERGICAL datasets — transactions, orders, customers, and catalogue — preparing them for PostgreSQL integration.

Operations performed: data type standardization, duplicate removal, handling of critical missing values, text normalization, and enrichment of transaction records via the product catalogue.

2. Load Raw Data

In [ ]:
import pandas as pd

In [ ]:
transactions=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\transactions_stage-V2.csv")
orders=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\orders_stage.csv")
customers=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\customers_stage-V2.csv")
catalogue=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\catalogue_stage.csv")
import pandas as pd

shipping_grid = pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\Frais de livraison NOEST .csv")

In [ ]:
datasets = {
    "Transactions": transactions,
    "Orders": orders,
    "Customers": customers,
    "Catalogue": catalogue,
}

3. Cleaning functions

In [ ]:
#cleaning data types
def clean_dtypes(df):
    for col in df.columns:
        if "date" in col:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    
    
    numeric_cols = ["quantity", "unit_price", "line_total",
                    "order_total_amount", "total_quantity", "n_lines","shipping_cost"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("string")

    return df


In [ ]:
#cleaning duplicates
def clean_duplicates(df):
    before=len(df)
    df = df.drop_duplicates()
    after=len(df)
    print(f"removed {before-after} duplicates")
    return df

In [ ]:
#cleaning missing values
def clean_missing_values(df, df_name):
    before=len(df)
    if df_name == "transactions":
        df = df.dropna(subset=["quantity", "unit_price", "line_total"])  
    after=len(df) 
    print(f"removed {before-after} incomplete rows")
    return df
        

In [ ]:
#Business rules
def validate_business_rules(df, df_name):

    print(f"\nBusiness Rule Validation - {df_name}")

    if "quantity" in df.columns:
        print(f"Negative quantities: {(df['quantity'] < 0).sum()}")

    if "unit_price" in df.columns:
        print(f"Negative prices: {(df['unit_price'] < 0).sum()}")
    if "line_total" in df.columns:
        print(f"Negative line totals: {(df['line_total'] < 0).sum()}")

    if "order_date" in df.columns:
        print(f"Future dates: {(df['order_date'] > pd.Timestamp.today()).sum()}")
    
    return df
        

In [ ]:
#Clean product name
def clean_product_names(df):

    if "product_name" in df.columns:

        df["product_name"] = (
            df["product_name"]
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    return df

4. Transactions Cleaning

In [ ]:
clean_transactions=transactions.copy()

In [ ]:
clean_transactions.drop(columns=["payment_method_group"], inplace=True)
clean_transactions.rename(
    columns={
        "Code Client": "customer_id_stage",
        "Titre moyen du paiement": "payment_method",
        "Titre de la méthode d’expédition": "shipping_method",
        "Montant total de la commande": "order_total_amount",
        "Montant de l’expédition commande": "shipping_cost",
        "Poids total": "total_weight"
    },
    inplace=True
)
clean_transactions["shipping_cost"] = (
    clean_transactions["shipping_cost"]
    .str.replace(" DA", "", regex=False)
    .str.replace(",", "", regex=False)
)
clean_transactions["order_total_amount"] = (
    clean_transactions["order_total_amount"]
    .str.replace(" DA", "", regex=False)
    .str.replace(",", "", regex=False)
)
clean_transactions=clean_dtypes(clean_transactions)


In [ ]:
clean_transactions["payment_method"] = clean_transactions["payment_method"].replace({
    "Autre": "Other",
    "Paiement à la livraison": "Cash on Delivery",
    "Versement CCP": "CCP Transfer",
    "Virement / Versement bancaire": "Bank Transfer",
    "Paiements par chèque": "Cheque Payment",
    "Paiement par chèque": "Cheque Payment",
    "Carte CIB & EDAHABIA": "CIB & Edahabia Card",
    "Carte CIB &amp; EDAHABIA": "CIB & Edahabia Card",
    "CIB / EDAHABIA": "CIB & Edahabia Card",
    "Espèces - Siège Social": "Cash - Siège Social",
    "Espèces - Bureau E-com": "Cash - E-commerce Office",
    "Espèces - Bureau Ecom": "Cash - E-commerce Office",
    "Espèces - Point d'enlèvement": "Cash - Pickup Point",
    "Other": "Other"
})

In [ ]:
clean_transactions=clean_duplicates(clean_transactions)
clean_transactions=clean_product_names(clean_transactions)
clean_transactions=clean_missing_values(clean_transactions,"transactions")
#Enrichment & merging 
clean_transactions = clean_transactions.merge(
    catalogue[["sku", "product_name", "subcategory"]],
    on="sku", how="left", suffixes=("", "_catalogue")
)
clean_transactions["product_name"] = clean_transactions["product_name"].fillna(clean_transactions["product_name_catalogue"])
clean_transactions["subcategory"] = clean_transactions["subcategory"].fillna(clean_transactions["subcategory_catalogue"])
clean_transactions["has_negative_price"] = clean_transactions["unit_price"] < 0
clean_transactions = clean_transactions.drop(columns=["product_name_catalogue", "subcategory_catalogue"])
print(f"Flagged as has_negative_price: {clean_transactions['has_negative_price'].sum()}")

clean_transactions["subcategory"] = clean_transactions["subcategory"].fillna("Unknown")
clean_transactions["product_name"] = clean_transactions["product_name"].fillna("Unknown")

In [ ]:
all_shipping_methods=clean_transactions["shipping_method"].drop_duplicates()
all_shipping_methods.to_csv("../../data/CleanData/all_shipping_methods.csv", index=False)



In [ ]:

clean_transactions["free_shipping"] = clean_transactions["shipping_method"].str.contains(
    "gratuit|gratuite|free|مجانا",
    case=False,
    na=False
)

In [ ]:
# Home Delivery
clean_transactions.loc[
    clean_transactions["shipping_method"].str.contains(
        "domicile|home|المنزل",
        case=False,
        na=False
    ),
    "shipping_method"
] = "Home Delivery"

In [ ]:
# Pickup Point
clean_transactions.loc[
    clean_transactions["shipping_method"].str.contains(
        "point|pick|نقطة",
        case=False,
        na=False
    ),
    "shipping_method"
] = "Pickup Point"

In [ ]:
# E-commerce Office
clean_transactions.loc[
    clean_transactions["shipping_method"].str.contains(
        "Bureau",
        case=False,
        na=False
    ),
    "shipping_method"
] = "E-commerce Office"

In [ ]:

# Customer Pickup
clean_transactions.loc[
    clean_transactions["shipping_method"].str.contains(
        "client",
        case=False,
        na=False
    ),
    "shipping_method"
] = "Customer Pickup" 
# International
clean_transactions.loc[
    clean_transactions["shipping_method"].str.contains(
        "International",
        case=False,
        na=False
    ),
    "shipping_method"
] = "International" 

In [ ]:
valid_methods = [
    "Home Delivery",
    "Pickup Point",
    "E-commerce Office",
    "Customer Pickup",
    "International"
]

clean_transactions.loc[
    ~clean_transactions["shipping_method"].isin(valid_methods),
    "shipping_method"
] = "Unknown"

In [ ]:
clean_transactions["payment_method"] = clean_transactions["payment_method"].fillna("Unknown")


5. Catalogue Cleaning

In [ ]:
clean_catalogue=catalogue.copy()
clean_catalogue=clean_dtypes(clean_catalogue)
clean_catalogue=clean_duplicates(clean_catalogue)
clean_catalogue=clean_product_names(clean_catalogue)

# Flag which catalogue products were actually sold
clean_catalogue["ever_sold"] = clean_catalogue["sku"].isin(clean_transactions["sku"].unique())

# Recover price for SKUs with transaction history
missing_price_skus = clean_catalogue[clean_catalogue["unit_price"].isna()]["sku"]

recovered_prices = (
    clean_transactions[clean_transactions["sku"].isin(missing_price_skus)]
    .groupby("sku")["unit_price"]
    .median()
)

clean_catalogue["unit_price"] = clean_catalogue["unit_price"].fillna(
    clean_catalogue["sku"].map(recovered_prices)
)

print(f"Remaining missing unit_price: {clean_catalogue['unit_price'].isna().sum()}")
clean_catalogue[]

6. Customers Cleaning

In [ ]:
clean_customers=customers.copy()
clean_customers=clean_duplicates(clean_customers)
clean_customers=clean_dtypes(clean_customers)

7. Orders Cleaning

In [ ]:
clean_orders=orders.copy()
clean_orders=clean_dtypes(clean_orders)
clean_orders=clean_duplicates(clean_orders)
clean_orders=clean_product_names(clean_orders)

8. Validation

In [ ]:
cleaned = {
    "Transactions": clean_transactions,
    "Orders": clean_orders,
    "Customers": clean_customers,
    "Catalogue": clean_catalogue
}

In [ ]:
for name, df in cleaned.items():
    print(name)
    print(df.info())
    print(df.isna().sum())
    print(df.duplicated().sum())
    validate_business_rules(df,name)

In [ ]:
clean_transactions["shipping_method"].value_counts(dropna=False)

In [ ]:
clean_transactions["payment_method"].value_counts(dropna=False)

9. Export Clean Datasets

In [ ]:
clean_transactions.to_csv("../../data/CleanData/clean_transactions.csv", index=False)
clean_orders.to_csv("../../data/CleanData/clean_orders.csv", index=False)
clean_customers.to_csv("../../data/CleanData/clean_customers.csv", index=False)
clean_catalogue.to_csv("../../data/CleanData/clean_catalogue.csv", index=False)


10. Cleaning Summary

In [ ]:
raw_datasets = {
    "Transactions": transactions,
    "Orders": orders,
    "Customers": customers,
    "Catalogue": catalogue,
}

summary_rows = []
for name, clean_df in cleaned.items():
    raw_rows = len(raw_datasets[name])
    clean_rows = len(clean_df)
    removed = raw_rows - clean_rows
    pct_removed = round((removed / raw_rows) * 100, 2)
    summary_rows.append({
        "Dataset": name,
        "Rows (raw)": raw_rows,
        "Rows (clean)": clean_rows,
        "Rows removed": removed,
        "% removed": pct_removed
    })

cleaning_summary = pd.DataFrame(summary_rows)
cleaning_summary


The data cleaning process focused on improving the consistency and reliability of the ENERGICAL datasets before database integration. The following operations were performed:

* Standardized data types (dates, numeric values, and categorical variables).
* Removed duplicate records.
* Removed records with missing values in critical business fields.
* Cleaned text fields.
* Standardized product names where applicable.
* Enriched transaction data using the product catalogue through SKU-based matching when reliable.
* Performed business rule validation to identify anomalies such as negative values and other records requiring manual review.
* Validated the cleaned datasets by checking data types, duplicates, missing values, and overall data integrity.

The cleaned datasets are now ready for PostgreSQL integration and subsequent analytical tasks. Detailed statistics, cleaning decisions, and identified data quality issues are documented in the accompanying Data Cleaning Report.
